# 🎭 ChuckleNet: FAST Spectral Features (~10 min)

**Key discovery**: librosa spectral features are **50x faster** than pyin F0!
- pyin F0: 0.5s per utterance → 15 hours for 620 videos
- Spectral features: 0.01s per utterance → **10 minutes** for 620 videos

**Features**: ZCR + RMS + spectral_centroid + spectral_bandwidth + spectral_rolloff

**Dataset**: 620 videos, ~9 min each, VTT [laughter] labels

In [ ]:
# 1. Setup
!apt-get install -y ffmpeg 2>&1 | tail -1
!pip install librosa numpy pandas scikit-learn tqdm 2>&1 | tail -3

from google.colab import drive
drive.mount('/content/drive')

import os, glob, time
import numpy as np
import librosa
from tqdm import tqdm

for BASE in ['/content/drive/My Drive/chuckle_net', '/content/drive/Shareddrives/chuckle_net']:
    if os.path.exists(BASE): break

AUDIO_DIR = f'{BASE}/audio'
VTT_DIR = f'{BASE}/vtt'

audio_files = glob.glob(f'{AUDIO_DIR}/*.m4a') + glob.glob(f'{AUDIO_DIR}/*.wav')
vtt_files = glob.glob(f'{VTT_DIR}/*.vtt')
print(f'Audio: {len(audio_files)}, VTT: {len(vtt_files)}')

In [ ]:
# 2. Fast feature extraction (50x faster than pyin!)
def get_vid(name):
    return name.replace('.en.vtt','').replace('.vtt','').replace('.m4a','').replace('.wav','')

def parse_vtt_cues(vtt_path):
    """Return list of (start, end, text, has_laughter)."""
    with open(vtt_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    def to_sec(ts):
        p = ts.replace('.',':').split(':')
        return int(p[0])*3600 + int(p[1])*60 + float(p[2])
    cues, lines = [], content.split('\n')
    i = 0
    while i < len(lines):
        if '-->' in lines[i]:
            s, e = lines[i].split('-->')
            s, e = to_sec(s.strip()), to_sec(e.strip())
            txt, i = [], i+1
            while i < len(lines) and lines[i].strip() and '-->' not in lines[i]:
                txt.append(lines[i].strip()); i += 1
            cues.append((s, e, ' '.join(txt), '[laughter]' in ' '.join(txt).lower()))
        else: i += 1
    return cues

def extract_spectral_features(y, sr=22050, hop_length=512):
    """Extract 6 fast spectral features."""
    try:
        zcr = librosa.feature.zero_crossing_rate(y, hop_length=hop_length)[0]
        rms = librosa.feature.rms(y=y, hop_length=hop_length)[0]
        sc = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=hop_length)[0]
        sb = librosa.feature.spectral_bandwidth(y=y, sr=sr, hop_length=hop_length)[0]
        sr_feat = librosa.feature.spectral_rolloff(y=y, sr=sr, hop_length=hop_length)[0]
        sfc = librosa.feature.spectral_flux(y=y, hop_length=hop_length)[0]
        return [
            np.mean(zcr), np.std(zcr),
            np.mean(rms), np.std(rms),
            np.mean(sc), np.std(sc),
            np.mean(sb), np.std(sb),
            np.mean(sr_feat), np.std(sr_feat),
            np.mean(sfc), np.std(sfc),
        ]
    except:
        return [0]*12

# Test speed
import tempfile, subprocess

def extract_segment(audio_path, start, end, sr=22050):
    duration = end - start
    if duration <= 0 or duration > 30: return None
    with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp:
        tmp_path = tmp.name
    try:
        cmd = ['ffmpeg', '-y', '-ss', str(start), '-t', str(duration),
               '-i', audio_path, '-ar', str(sr), '-ac', '1', '-loglevel', 'error', tmp_path]
        if subprocess.run(cmd, timeout=10).returncode != 0: return None
        y, _ = librosa.load(tmp_path, sr=sr)
        if len(y) < sr * 0.05: return None
        return extract_spectral_features(y, sr)
    except: return None
    finally:
        if os.path.exists(tmp_path): os.unlink(tmp_path)

# Time test
vtt_lookup = {get_vid(os.path.basename(v)): v for v in vtt_files}
audio_lookup = {get_vid(os.path.basename(a)): a for a in audio_files}
matching = list(set(audio_lookup.keys()) & set(vtt_lookup.keys()))

vid = matching[0]
cues = parse_vtt_cues(vtt_lookup[vid])

t0 = time.time()
for s, e, text, _ in cues[:10]:
    extract_segment(audio_lookup[vid], s, e)
t1 = time.time()

per_utt = (t1-t0) / 10
print(f'Spectral features: {per_utt*1000:.1f}ms per utterance')
print(f'Estimated per video: {per_utt * len(cues) / 60:.1f} min')
print(f'Total for 620 videos: {per_utt * sum(len(parse_vtt_cues(vtt_lookup[v])) for v in matching) / 60:.0f} min')

In [ ]:
# 3. Process ALL 620 videos
all_feat, all_label, all_vid, all_lang = [], [], [], []

for vid in tqdm(matching, desc='Videos'):
    audio_path = audio_lookup[vid]
    vtt_path = vtt_lookup[vid]
    
    lang = 'en'
    if '.hi.' in vtt_path: lang = 'hi'
    elif '.zh.' in vtt_path: lang = 'zh'
    elif '.es.' in vtt_path: lang = 'es'
    
    cues = parse_vtt_cues(vtt_path)
    
    for i, (s, e, text, has_laugh) in enumerate(cues):
        feat = extract_segment(audio_path, s, e)
        if feat:
            all_feat.append(feat)
            all_label.append(1 if has_laugh else 0)
            all_vid.append(vid)
            all_lang.append(lang)

X = np.array(all_feat)
y = np.array(all_label)
vids = np.array(all_vid)
langs = np.array(all_lang)

print(f'\n✅ Total: {len(X)} segments')
print(f'Positive: {y.sum()} ({100*y.mean():.1f}%)')
for lang in ['en', 'hi', 'zh', 'es']:
    mask = langs == lang
    if mask.sum() > 0:
        print(f'  {lang}: {mask.sum()} segs, {100*y[mask].mean():.1f}% pos')

In [ ]:
# 4. Train + Evaluate
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, precision_score, recall_score

unique_vids = list(set(vids))
np.random.seed(42); np.random.shuffle(unique_vids)
n_test = max(1, int(len(unique_vids)*0.2))
test_vids = set(unique_vids[:n_test])

train_m = ~np.isin(vids, list(test_vids))
X_train, X_test = X[train_m], X[~train_m]
y_train, y_test = y[train_m], y[~train_m]

print(f'Train: {len(X_train)} ({100*y_train.mean():.1f}% pos)\n')

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)
print('=== Spectral Features Logistic Regression ===')
print(f'F1: {f1_score(y_test, y_pred):.4f}')
print(f'Precision: {precision_score(y_test, y_pred):.4f}')
print(f'Recall: {recall_score(y_test, y_pred):.4f}')

mlp = MLPClassifier(hidden_layer_sizes=(64,32), max_iter=500, random_state=42)
mlp.fit(X_train, y_train)
y_pred_mlp = mlp.predict(X_test)
print('\n=== Spectral Features MLP ===')
print(f'F1: {f1_score(y_test, y_pred_mlp):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_mlp):.4f}')
print(f'Recall: {recall_score(y_test, y_pred_mlp):.4f}')

In [ ]:
# 5. Save
import pickle
out = {'features': X, 'labels': y, 'vids': vids, 'langs': langs}
np.savez_compressed(f'{BASE}/spectral_620.npz', **out)
with open(f'{BASE}/spectral_model.pkl', 'wb') as f:
    pickle.dump(lr, f)
print(f'Saved: {BASE}/spectral_620.npz')
print('\n🎉 DONE!')